## Importation des bibliothèques

In [1]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import time

## Importation de la base de données

In [ ]:
# Récupération des données via l'url accessible sur : https://www.data.gouv.fr/datasets/recensement-des-equipements-sportifs-espaces-et-sites-de-pratiques
# Chargement long

# data = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/ea4f5879-af40-4e3e-949d-812d6eeb5e02")

# Récupération directement depuis le dernier csv téléchargé au préalable et stocké sur le s3

data = pd.read_csv("https://minio.lab.sspcloud.fr/matteo/data-es.csv", sep=";")


data_nautique = data[data['equip_type_famille'] == "Site d'activités aquatiques et nautiques"]

colonnes = ['equip_numero', "inst_numero", "inst_nom", "inst_adresse", "inst_cp", "dep_code", "reg_code", "dep_nom", "reg_nom", "lib_bdv", "equip_nom", "equip_type_name", "equip_coordonnees", "aps_name"]
data_nautique = data_nautique[colonnes].reset_index(drop=True).copy()
mots = ["surf", "ski", "voile", "foil", "wake"]
pattern = "|".join(mots)
data_nautique = data_nautique[data_nautique['aps_name'].str.contains(pattern, case=False, na=False)]
data_nautique['dep_code'] = data_nautique['dep_code'].astype(str)
data_nautique

## Fonctions récupération API

In [ ]:
# ── Setup session avec cache 1h et retry automatique ──────────────
cache_session = requests_cache.CachedSession('.cache_meteo', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# ── Variables météo à récupérer ───────────────────────────────────
VARIABLES_HORAIRES = [
    "temperature_2m",          # Température air (°C)
    "relative_humidity_2m",     # Humidité relative (%)
    "apparent_temperature",     # Température ressentie (°C)
    "precipitation",            # Précipitations (mm)
    "wind_speed_10m",           # Vitesse vent à 10m (km/h)
    "wind_gusts_10m",           # Rafales (km/h)
    "wind_direction_10m",       # Direction vent (°)
    "weather_code",             # Code météo WMO
    "cloud_cover",              # Couverture nuageuse (%)
    "pressure_msl",             # Pression mer (hPa)
    "shortwave_radiation",      # Rayonnement solaire (W/m²)
    "visibility",               # Visibilité (m) — important côtier
]

VARIABLES_JOURNALIERES = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "wind_direction_10m_dominant",
    "sunrise",
    "sunset",
    "uv_index_max",
    "weather_code",
]

# ── Récupération des données ──────────────────────────────────────
def fetch_meteo_departement(nom, lat, lon, facade):
    params = {
        "latitude":             lat,
        "longitude":            lon,
        "hourly":               VARIABLES_HORAIRES,
        "daily":                VARIABLES_JOURNALIERES,
        "current": ["temperature_2m", "wind_speed_10m",
                     "wind_gusts_10m", "precipitation",
                     "weather_code", "pressure_msl"],
        "models":               "meteofrance_seamless",
        "timezone":             "Europe/Paris",
        "forecast_days":        4,
        "wind_speed_unit":      "kmh",
        "temperature_unit":     "celsius",
        "precipitation_unit":   "mm",
        "temporal_resolution":  "hourly_2",  # ← 1 valeur toutes les 2h
    }
    url = "https://api.open-meteo.com/v1/meteofrance"
    responses = openmeteo.weather_api(url, params=params)
    r = responses[0]

    h = r.Hourly()
    times = pd.date_range(
        start=pd.to_datetime(h.Time(), unit="s", utc=True),
        end=pd.to_datetime(h.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=h.Interval()),
        inclusive="left"
    ).tz_convert("Europe/Paris")

    df_h = pd.DataFrame({"time": times})
    for i, var in enumerate(VARIABLES_HORAIRES):
        df_h[var] = h.Variables(i).ValuesAsNumpy()
    df_h["departement"] = nom
    df_h["facade"]      = facade
    df_h["latitude"]    = lat
    df_h["longitude"]   = lon

    return df_h

## Filtration des équipement par demande

In [ ]:
from math import radians, sin, cos, sqrt, atan2

# ── Fonction distance Haversine ───────────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# ── Filtre par ville et rayon ─────────────────────────────────────
def filter_by_ville(df, nom_ville, rayon_km):
    """
    df        : DataFrame source contenant equip_coordonnees et lib_bdv
    nom_ville : Nom de la ville (colonne lib_bdv)
    rayon_km  : Rayon de recherche en kilomètres
    """
    # Récupère les coordonnées de la ville
    ville_rows = df[df["lib_bdv"].str.lower() == nom_ville.lower()]
    
    if ville_rows.empty:
        print(f"⚠️  Ville '{nom_ville}' introuvable dans le dataframe.")
        print("Villes disponibles :", sorted(df["lib_bdv"].dropna().unique().tolist()))
        return None

    lat_ville = ville_rows["latitude"].iloc[0]
    lon_ville = ville_rows["longitude"].iloc[0]
    print(f"✓ Ville trouvée : {nom_ville} ({lat_ville}, {lon_ville})")

    # Calcule la distance de chaque équipement par rapport à la ville
    df = df.copy()
    df["distance_km"] = df.apply(
        lambda row: haversine(lat_ville, lon_ville, row["latitude"], row["longitude"]),
        axis=1
    )

    # Filtre par rayon
    df_filtre = df[df["distance_km"] <= rayon_km].copy()
    df_filtre = df_filtre.sort_values("distance_km").reset_index(drop=True)

    print(f"✓ {len(df_filtre)} équipements dans un rayon de {rayon_km} km autour de {nom_ville}")
    return df_filtre

df_source = data_nautique.copy()

# ── Utilisation ───────────────────────────────────────────────────
# Assure-toi que les colonnes latitude/longitude sont bien présentes
df_source[["latitude", "longitude"]] = (
    df_source["equip_coordonnees"]
    .str.split(",", expand=True)
    .astype(float)
)

# Exemple : équipements dans un rayon de 50 km autour de Brest
df_filtre = filter_by_ville(df_source, nom_ville="Brest", rayon_km=20)

# Aperçu du résultat
if df_filtre is not None:
    print(df_filtre[["equip_nom", "lib_bdv", "dep_nom", "distance_km"]].head(10).to_string(index=False))

## Récupération des données météo avec l'API

In [ ]:
# ── Boucle de récupération météo ──────────────────────────────────
all_dfs = []
for _, row in df_filtre.iterrows():
    nom    = row["dep_nom"]
    lat    = row["latitude"]
    lon    = row["longitude"]
    facade = "Inconnue"

    print(f"Récupération : {nom}...")
    df = fetch_meteo_departement(nom, lat, lon, facade)

    # Propagation des colonnes utiles
    df["dep_nom"]         = row["dep_nom"]
    df["dep_code"]        = row["dep_code"]
    df["reg_nom"]         = row["reg_nom"]
    df["equip_numero"]    = row["equip_numero"]
    df["equip_nom"]       = row["equip_nom"]
    df["equip_type"]      = row["equip_type_name"]
    df["inst_nom"]        = row["inst_nom"]
    df["inst_adresse"]    = row["inst_adresse"]
    df["aps_name"]        = row["aps_name"]

    all_dfs.append(df)
    time.sleep(0.1)

df_cotes = pd.concat(all_dfs, ignore_index=True)
print(f"\n✓ {len(df_cotes)} lignes — {df_cotes['dep_nom'].nunique()} départements")

## Exportation des données méteo

In [ ]:
# ── Export ───────────────────────────────────────────────────────

# Supprime la timezone pour compatibilité Excel
df_export = df_cotes.copy()
df_export["time"] = df_export["time"].dt.tz_localize(None)

# ── CSV ──────────────────────────────────────────────────────────
df_export.to_csv("meteo_cotes_france.csv", index=False, encoding="utf-8-sig")
print("✓ CSV exporté : meteo_cotes_france.csv")

# ── Conditions actuelles par département ─────────────────────────
now_df = df_export[df_export["time"] == df_export.groupby("dep_nom")["time"].transform("min")]
print("\n── Conditions actuelles ──")
print(now_df[["dep_nom", "equip_nom", "temperature_2m",
              "wind_speed_10m", "wind_gusts_10m", "precipitation"]].to_string(index=False))

# ── Vent max par département ──────────────────────────────────────
vent_max = (df_export
    .groupby("dep_nom")["wind_gusts_10m"]
    .max()
    .sort_values(ascending=False)
    .reset_index())
vent_max.columns = ["Département", "Rafales max (km/h)"]
print("\n── Rafales max par département ──")
print(vent_max.to_string(index=False))

# ── Alerte pluie : équipements > 5mm sur 4 jours ─────────────────
pluie_alert = (df_export
    .groupby(["dep_nom", "equip_nom"])["precipitation"]
    .sum()
    .reset_index()
    .query("precipitation > 5")
    .sort_values("precipitation", ascending=False))
pluie_alert.columns = ["Département", "Équipement", "Précipitations totales (mm)"]
print("\n── Équipements avec pluie > 5mm ──")
print(pluie_alert.to_string(index=False))

# ── Export Excel avec une feuille par département ─────────────────
with pd.ExcelWriter("meteo_cotes_france.xlsx", engine="openpyxl") as writer:
    for dep, group in df_export.groupby("dep_nom"):
        sheet_name = dep[:31]  # Excel limite les noms d'onglets à 31 caractères
        group.to_excel(writer, sheet_name=sheet_name, index=False)
print("\n✓ Excel exporté : meteo_cotes_france.xlsx")

## Mise en place du Score 

In [ ]:
df = pd.read_csv("meteo_cotes_france.csv")
df

In [ ]:
def score_surf(wind_speed, wind_direction, temperature, wind_gust):
    score = 0

    # --- 🌬️ Vent (vitesse) ---
    # Idéal : vent faible à modéré
    if wind_speed < 5:
        score += 2
    elif 5 <= wind_speed <= 15:
        score += 4
    elif 15 < wind_speed <= 25:
        score += 1
    else:
        score -= 3

    # --- 🧭 Direction du vent ---
    # offshore = excellent, onshore = mauvais
    offshore = ["E", "NE", "SE"]   # dépend de la côte, ici simplifié
    onshore = ["W", "NW", "SW"]

    if wind_direction in offshore:
        score += 4
    elif wind_direction in onshore:
        score -= 4
    else:
        score += 1  # vent cross-shore

    # --- 🌡️ Température ---
    if temperature >= 25:
        score += 3
    elif 18 <= temperature < 25:
        score += 2
    elif 10 <= temperature < 18:
        score += 0
    else:
        score -= 2

    # --- 💨 Rafales ---
    # Trop de rafales = mer désordonnée
    if wind_gust - wind_speed < 5:
        score += 2
    elif wind_gust - wind_speed < 15:
        score += 0
    else:
        score -= 3

    # --- Normalisation (optionnel) ---
    # On borne le score
    score = max(0, min(score, 10))

    return score

df["score_surf"] = df.apply(
    lambda row: score_surf(
        row["wind_speed_10m"],
        row["wind_direction_10m"],
        row["temperature_2m"],
        row["wind_gusts_10m"]
    ),
    axis=1
)


def score_wakeboard(wind_speed, temperature, wind_gust):
    score = 0

    # 🌬️ Vent (trop de vent = clapot)
    if wind_speed < 10:
        score += 4
    elif 10 <= wind_speed <= 20:
        score += 2
    else:
        score -= 3

    # 💨 Rafales
    if wind_gust - wind_speed < 5:
        score += 3
    else:
        score -= 2

    # 🌡️ Température
    if temperature >= 25:
        score += 3
    elif temperature >= 18:
        score += 2
    elif temperature >= 10:
        score += 0
    else:
        score -= 3

    return max(0, min(score, 10))

df["score_wakeboard"] = df.apply(
lambda row: score_wakeboard(
    row["wind_speed_10m"],
    row["temperature_2m"],
    row["wind_gusts_10m"]
    ),
    axis=1
)



def score_foil(wind_speed, wind_direction, wind_gust):
    score = 0

    # 🌬️ Vent idéal faible
    if 8 <= wind_speed <= 15:
        score += 5
    elif 5 <= wind_speed < 8 or 15 < wind_speed <= 20:
        score += 2
    else:
        score -= 3

    # 💨 Rafales (très pénalisantes en foil)
    if wind_gust - wind_speed < 5:
        score += 3
    else:
        score -= 3

    # 🧭 Direction (comme surf simplifié)
    offshore = ["E", "NE", "SE"]
    onshore = ["W", "NW", "SW"]

    if wind_direction in offshore:
        score += 2
    elif wind_direction in onshore:
        score -= 2


    return max(0, min(score, 10))

df["score_foil"] = df.apply(
lambda row: score_wakeboard(
    row["wind_speed_10m"],
    row["wind_direction_10m"],
    row["wind_gusts_10m"]),
    axis=1
)


def score_char_a_voile(wind_speed, wind_gust):
    score = 0

    # 🌬️ Vent (clé principale)
    if 15 <= wind_speed <= 30:
        score += 5
    elif 10 <= wind_speed < 15:
        score += 3
    elif wind_speed < 10:
        score -= 3
    else:
        score += 2  # très fort vent mais praticable

    # 💨 Rafales (peu gênantes mais dangereuses)
    if wind_gust - wind_speed < 10:
        score += 2
    else:
        score -= 2

    return max(0, min(score, 10))

df["score_char_a_voile"] = df.apply(
lambda row: score_char_a_voile(
    row["wind_speed_10m"],
    row["wind_gusts_10m"]),
    axis=1
)



def score_planche_a_voile(wind_speed, wind_direction, wind_gust):
    score = 0

    # 🌬️ Vent
    if 12 <= wind_speed <= 25:
        score += 5
    elif 8 <= wind_speed < 12:
        score += 2
    elif wind_speed > 25:
        score += 1
    else:
        score -= 3

    # 💨 Rafales
    if wind_gust - wind_speed < 7:
        score += 2
    else:
        score -= 2

    # 🧭 Direction
    offshore = ["E", "NE", "SE"]
    onshore = ["W", "NW", "SW"]

    if wind_direction in offshore:
        score += 2
    elif wind_direction in onshore:
        score += 1  # contrairement au surf, c'est OK

    return max(0, min(score, 10))

df["score_planche_a_voile"] = df.apply(
lambda row: score_planche_a_voile(
    row["wind_speed_10m"],
    row["wind_direction_10m"],
    row["wind_gusts_10m"]),
    axis=1
)  

df
